In [2]:
import requests
import pandas as pd
from datetime import datetime


headers = {
    "X-RapidAPI-Key": "4005866697msh2b2c979bf694ec1p1cef3fjsne0259089faa3",
    "X-RapidAPI-Host": "cricbuzz-cricket.p.rapidapi.com"
}

url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/live"

response = requests.get(url, headers=headers)
data = response.json()

In [80]:
from sqlalchemy import create_engine

server = 'DESKTOP-B66UO89\\SQLEXPRESS'
database = 'CricketDB'

connection_string = (
    "mssql+pyodbc://@DESKTOP-B66UO89\\SQLEXPRESS/CricketDB?"
    "driver=ODBC+Driver+17+for+SQL+Server&"
    "trusted_connection=yes"
)


In [6]:
## Get live match data (1st page)

In [3]:
rows = []

for mt in data.get('typeMatches', []):
    for series in mt.get('seriesMatches', []):
        wrapper = series.get('seriesAdWrapper')
        if wrapper:
            for match in wrapper.get('matches', []):
                info = match['matchInfo']
                rows.append({
                    "match_id": info.get("matchId"),
                    "series_name": info.get("seriesName"),
                    "format": info.get("matchFormat"),
                    "team1_name": info["team1"]["teamName"],
                    "team2_name": info["team2"]["teamName"],
                    "status": info.get("status"),
                    "venue_name": info["venueInfo"]["city"],
                    "startdate": datetime.fromtimestamp(int(info['seriesStartDt'])/1000),
                    "enddate": datetime.fromtimestamp(int(info['seriesEndDt'])/1000)
                })
matches_live = pd.DataFrame(rows)

In [22]:
matches_live.to_sql('matches_live', con=connection_string, if_exists='replace', index=False)

9

In [5]:
matches_live

,match_id,series_name,format,team1_name,team2_name,status,venue_name,startdate,enddate
0,148503,"Sweden tour of Indonesia, 2026",T20,Sweden,Indonesia,Sweden opt to bat,Bali,2026-04-07 05:30:00,2026-04-15 05:30:00
1,148492,"Sweden tour of Indonesia, 2026",T20,Sweden,Indonesia,Indonesia won by 4 wkts,Bali,2026-04-07 05:30:00,2026-04-15 05:30:00
2,142098,County Championship Division One 2026,TEST,Hampshire,Yorkshire,Day 3: Stumps - Yorkshire need 361 runs,Leeds,2026-04-03 05:30:00,2026-09-29 05:30:00
3,142109,County Championship Division One 2026,TEST,Surrey,Leicestershire,Day 3: Stumps - Leicestershire lead by 171 runs,London,2026-04-03 05:30:00,2026-09-29 05:30:00
4,142087,County Championship Division One 2026,TEST,Warwickshire,Sussex,Day 3: Stumps - Sussex need 94 runs,Hove,2026-04-03 05:30:00,2026-09-29 05:30:00
5,142120,County Championship Division One 2026,TEST,Nottinghamshire,Glamorgan,Day 3: Stumps - Glamorgan need 346 runs,Nottingham,2026-04-03 05:30:00,2026-09-29 05:30:00
6,142803,County Championship Division Two 2026,TEST,Northamptonshire,Kent,Day 3: Stumps - Kent trail by 344 runs,Canterbury,2026-04-03 05:30:00,2026-09-29 05:30:00
7,142814,County Championship Division Two 2026,TEST,Lancashire,Derbyshire,Day 3: Stumps - Lancashire lead by 124 runs,Manchester,2026-04-03 05:30:00,2026-09-29 05:30:00
8,153039,"Women's Under-19 Tri-Series in Australia, 2026",T20,Australia Women U19,Sri Lanka Women U19,Sri Lanka Women U19 won by 4 wkts,Gold Coast,2026-04-06 05:30:00,2026-04-20 05:30:00


## 2nd top_scores_details

In [82]:
all_matches_players = []

match_ids = matches_live["match_id"]

for mid in match_ids:
    score_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{mid}/scard"
    res = requests.get(score_url, headers=headers)

    if res.status_code == 200:
        score_data = res.json()

        for inning in score_data.get('scorecard', []):
                    # -------------------------
            # 🏏 BATSMEN DATA
            # -------------------------
            for b in inning.get('batsman', []):
                all_matches_players.append({
                    "player_id": b.get('id'),
                    "name": b.get('name'),
                    "role": "batsman",
                    "match_id": mid,
                    "runs": b.get('runs'),
                    "balls": b.get('balls'),
                    "fours": b.get('fours'),
                    "sixes": b.get('sixes'),
                    "strike_rate": b.get('strkrate'),
                    "dismissal": b.get('outdec'),
                    "innings": inning.get('inningsid'),
                    "team_wickets": team_wickets,
                    "team_overs": team_overs,
        
                    })
        
                    # -------------------------
                    # 🎯 BOWLERS DATA (IMPORTANT)
                    # -------------------------
            for bowler in inning.get('bowler', []):
                all_matches_players.append({
                    "player_id": bowler.get('id'),
                    "name": bowler.get('name'),
                    "role": "bowler",
                    "match_id": mid,
                    "overs": bowler.get('overs'),
                    "maidens": bowler.get('maidens'),
                    "runs_conceded": bowler.get('runs'),
                    "wickets": bowler.get('wickets'),
                    "economy": bowler.get('economy'),
                    "innings": inning.get('inningsid'),
                    "team_wickets": team_wickets,
                    "team_overs": team_overs,
                        }) 
    else:
        print(f"Failed for match {mid}: {res.status_code}")

In [83]:
players = pd.DataFrame(all_matches_players)

In [84]:
players

,player_id,name,role,match_id,runs,balls,fours,sixes,strike_rate,dismissal,innings,team_wickets,team_overs,overs,maidens,runs_conceded,wickets,economy
0,28649,Imal Zuwak,batsman,148503,5.0,3.0,1.0,0.0,166.67,b Gaurav Tiwari,1,6,19.3,NaN,NaN,NaN,NaN,NaN
1,44504,Ajay Mundra,batsman,148503,6.0,3.0,0.0,1.0,200,c Gede Priandana b Ferdinando Banunaek,1,6,19.3,NaN,NaN,NaN,NaN,NaN
2,1456633,Awais Ahmad,batsman,148503,0.0,1.0,0.0,0.0,0,c Danilson Hawoe b Ferdinando Banunaek,1,6,19.3,NaN,NaN,NaN,NaN,NaN
3,1461378,Yatharth Chauhan,batsman,148503,32.0,28.0,4.0,1.0,114.29,c Gaurav Tiwari b Kadek Gamantika,1,6,19.3,NaN,NaN,NaN,NaN,NaN
4,26386,Wynand Boshoff,batsman,148503,0.0,3.0,0.0,0.0,0,b Gaurav Tiwari,1,6,19.3,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
442,1470437,Aurora Mavros,bowler,153039,NaN,NaN,NaN,NaN,NaN,NaN,2,6,19.3,3,0.0,15.0,0.0,5
443,1463782,Filippa Suesee,bowler,153039,NaN,NaN,NaN,NaN,NaN,NaN,2,6,19.3,4,0.0,18.0,3.0,4.5
444,1448981,Tegan Williamson,bowler,153039,NaN,NaN,NaN,NaN,NaN,NaN,2,6,19.3,4,0.0,19.0,0.0,4.8
445,1463848,Emily Powell,bowler,153039,NaN,NaN,NaN,NaN,NaN,NaN,2,6,19.3,1,0.0,5.0,0.0,5


In [85]:
players.to_sql('players', con=connection_string, if_exists='replace', index=False)

99

In [87]:
top_runs_per_match = (players.sort_values(["match_id", "runs"], ascending=[True, False]).groupby("match_id").first().reset_index()[["match_id", "name", "runs"]])

In [88]:
Highest_score_per_match = (players.sort_values(["match_id", "runs"], ascending=[True, False]).groupby("match_id").first().reset_index()[["match_id", "name", "runs","balls"]])

In [89]:
top_strikerate_per_match = (players.sort_values(["match_id", "strike_rate","runs"], ascending=[True, False,False]).groupby("match_id").first().reset_index()[["match_id", "name", "strike_rate","runs"]])

In [90]:
most_wickets_per_match = (players.sort_values(["match_id", "team_wickets"], ascending=[True, False]).groupby("match_id").first().reset_index()[["match_id", "name", "wickets","overs"]])

In [93]:
top_runs_per_match 

,match_id,name,runs
0,142087,Robert Yates,90.0
1,142098,Ben Brown,103.0
2,142109,Jamie Smith,166.0
3,142120,Joe Clarke,136.0
4,142803,Luke Procter,261.0
5,142814,Marcus Harris,125.0
6,148492,Kavin Neeraj Chaddha,60.0
7,148503,Saeed Ahmed,38.0
8,153039,Pramudi Methsara,21.0


In [92]:
top_strikerate_per_match

,match_id,name,strike_rate,runs
0,142087,James Coles,91.67,11.0
1,142098,Ben Brown,94.5,103.0
2,142109,Dan Lawrence,86.11,31.0
3,142120,Kyle Verreynne,93.33,14.0
4,142803,Tawanda Muyeye,84.21,48.0
5,142814,Anuj Dal,95.45,21.0
6,148492,Advait Dhabe,87.5,21.0
7,148503,Apriliandi Rahayu,366.67,11.0
8,153039,Indira Panelli,87.5,7.0


In [7]:
# import requests
# import pandas as pd
# from datetime import datetime

# headers = {
#     "X-RapidAPI-Key": "39902b1d99mshd7ca7a0ef0d9f74p1b0117jsnf7edb4c7a10b",
#     "X-RapidAPI-Host": "cricbuzz-cricket.p.rapidapi.com"
# }

# url1 = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/recent"

# responser = requests.get(url1, headers=headers)
# datar = responser.json()

## SQL questions

In [9]:
# import pandas as pd

# def flatten_match(datar):
#     matches = datar.get("typeMatches", [])

#     rows = []

#     for mt in matches:
#         match_type = mt.get("matchType")

#         for series in mt.get("seriesMatches", []):
#             series_info = series.get("seriesAdWrapper", {})

#             for match in series_info.get("matches", []):

#                 m = match.get("matchInfo", {})
#                 score = match.get("matchScore", {})

#                 row = {
#                     # 🏏 match info
#                     "match_id": m.get("matchId"),
#                     "series_id": m.get("seriesId"),
#                     "series_name": m.get("seriesName"),
#                     "match_desc": m.get("matchDesc"),
#                     "match_format": m.get("matchFormat"),
#                     "start_date": datetime.fromtimestamp(int(m.get("startDate"))/1000),
#                     "end_date": datetime.fromtimestamp(int(m.get("endDate"))/1000),
#                     "state": m.get("state"),
#                     "status": m.get("status"),

#                     # 🏟️ teams
#                     "team1_id": m.get("team1", {}).get("teamId"),
#                     "team1_name": m.get("team1", {}).get("teamName"),
#                     "team1_short": m.get("team1", {}).get("teamSName"),

#                     "team2_id": m.get("team2", {}).get("teamId"),
#                     "team2_name": m.get("team2", {}).get("teamName"),
#                     "team2_short": m.get("team2", {}).get("teamSName"),

#                     # 📊 scores
#                     "team1_runs": score.get("team1Score", {}).get("inngs1", {}).get("runs"),
#                     "team1_wkts": score.get("team1Score", {}).get("inngs1", {}).get("wickets"),
#                     "team1_overs": score.get("team1Score", {}).get("inngs1", {}).get("overs"),

#                     "team2_runs": score.get("team2Score", {}).get("inngs1", {}).get("runs"),
#                     "team2_wkts": score.get("team2Score", {}).get("inngs1", {}).get("wickets"),
#                     "team2_overs": score.get("team2Score", {}).get("inngs1", {}).get("overs"),

#                     # 📍 venue
#                     "venue_id": m.get("venueInfo", {}).get("id"),
#                     "ground": m.get("venueInfo", {}).get("ground"),
#                     "city": m.get("venueInfo", {}).get("city"),

#                     # 🧠 result
#                     "result": m.get("status")
#                 }

#                 rows.append(row)

#     return pd.DataFrame(rows)

In [10]:
df = flatten_match(datar)

In [11]:
df

,match_id,series_id,series_name,match_desc,match_format,start_date,end_date,state,status,team1_id,...,team1_runs,team1_wkts,team1_overs,team2_runs,team2_wkts,team2_overs,venue_id,ground,city,result
0,151517,7572,ICC Cricket World Cup League Two 2023-27,97th Match,ODI,2026-04-12 13:00:00,2026-04-12 21:00:00,Complete,Scotland won by 7 wkts,161,...,198,10,48.6,199,3,37.6,1438021,Namibia Cricket Ground,Windhoek,Scotland won by 7 wkts
1,151506,7572,ICC Cricket World Cup League Two 2023-27,96th Match,ODI,2026-04-10 13:00:00,2026-04-10 21:00:00,Complete,Namibia won by 1 wkt,304,...,218,9,49.6,223,9,49.3,363,Wanderers Cricket Ground,Windhoek,Namibia won by 1 wkt
2,148503,11652,"Sweden tour of Indonesia, 2026",8th T20I,T20,2026-04-13 11:30:00,2026-04-13 15:00:00,Complete,Sweden won by 18 runs,534,...,166,8,19.6,148,10,18.2,1179,Udayana Cricket Ground,Bali,Sweden won by 18 runs
3,148492,11652,"Sweden tour of Indonesia, 2026",7th T20I,T20,2026-04-13 07:00:00,2026-04-13 10:30:00,Complete,Indonesia won by 4 wkts,534,...,145,9,19.6,148,6,17.4,1179,Udayana Cricket Ground,Bali,Indonesia won by 4 wkts
4,148481,11652,"Sweden tour of Indonesia, 2026",6th T20I,T20,2026-04-11 11:30:00,2026-04-11 15:00:00,Complete,Sweden won by 87 runs,534,...,170,6,19.6,83,10,15.2,1179,Udayana Cricket Ground,Bali,Sweden won by 87 runs
5,148470,11652,"Sweden tour of Indonesia, 2026",5th T20I,T20,2026-04-11 07:00:00,2026-04-11 10:30:00,Complete,Sweden won by 5 wkts,566,...,165,5,19.6,166,5,18.6,1179,Udayana Cricket Ground,Bali,Sweden won by 5 wkts
6,148459,11652,"Sweden tour of Indonesia, 2026",4th T20I,T20,2026-04-09 11:30:00,2026-04-09 15:00:00,Complete,Sweden won by 10 runs,534,...,164,7,19.6,154,9,19.6,1179,Udayana Cricket Ground,Bali,Sweden won by 10 runs
7,148874,11704,"Scotland tour of Namibia, 2026",1st T20I,T20,2026-04-15 17:30:00,2026-04-15 21:00:00,Complete,Scotland won by 7 wkts,161,...,159,7,19.6,160,3,18.5,1438021,Namibia Cricket Ground,Windhoek,Scotland won by 7 wkts
8,150285,11740,"Portugal T20I Tri-Series, 2026",Final,T20,2026-04-09 19:00:00,2026-04-09 22:30:00,Complete,Portugal won by 30 runs,596,...,183,6,19.6,153,10,18.6,1528,Santarem Cricket Ground,Albergaria,Portugal won by 30 runs
9,150274,11740,"Portugal T20I Tri-Series, 2026",9th Match,T20,2026-04-09 14:30:00,2026-04-09 18:00:00,Complete,France won by 4 wkts,1027,...,111,10,18.5,112,6,17.1,1528,Santarem Cricket Ground,Albergaria,France won by 4 wkts


In [20]:
df.columns

Index(['match_id', 'series_id', 'series_name', 'match_desc', 'match_format',
       'start_date', 'end_date', 'state', 'status', 'team1_id', 'team1_name',
       'team1_short', 'team2_id', 'team2_name', 'team2_short', 'team1_runs',
       'team1_wkts', 'team1_overs', 'team2_runs', 'team2_wkts', 'team2_overs',
       'venue_id', 'ground', 'city', 'result'],
      dtype='object')

## Matches

In [97]:
matches = df[['match_desc','match_id','team1_name','team2_name','venue_id','ground','city','match_format','start_date']].drop_duplicates(ignore_index=True)

In [99]:
matches.to_sql('matches', con=connection_string, if_exists='replace', index=False)

52

## Series

In [91]:
series = df[['series_id', 'series_name','match_format','start_date','end_date']].drop_duplicates(ignore_index=True)

In [94]:
series.to_sql('series', con=connection_string, if_exists='replace', index=False)

45

## Venues

In [41]:
venues = []

venueId = df["venue_id"].dropna().unique()


headers = {
    "X-RapidAPI-Key": "1b7742aaf8mshce1201037d8658ap158288jsn90656ad61c14",
    "X-RapidAPI-Host": "cricbuzz-cricket.p.rapidapi.com"
}

for vid in venueId:
    url = f"https://cricbuzz-cricket.p.rapidapi.com/venues/v1/{int(vid)}"
    response = requests.get(url, headers=headers)
    data = response.json()
    venues.append({"venue_id": int(vid),
                    "venue_name": data.get("ground"),
                   "city": data.get("city"),
                    "country": data.get("country"),
                   "capacity": data.get("capacity")})


In [ ]:
import json
import os

CACHE_FILE = "venues_cache.json"

# load cache
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        cache = json.load(f)
else:
    cache = {}

def get_venue(vid):
    if str(vid) in cache:
        return cache[str(vid)]  # ✅ no API call

    url = f"https://cricbuzz-cricket.p.rapidapi.com/venues/get-info/venues/v1/{vid}"
    res = requests.get(url, headers=headers)

    if res.status_code == 200:
        data = res.json()
        cache[str(vid)] = data

        # save immediately
        with open(CACHE_FILE, "w") as f:
            json.dump(cache, f)

        return data

In [43]:
venues_details = pd.DataFrame(venues)

In [81]:
venues_details.to_sql('venues', con=connection_string, if_exists='replace', index=False)

27

In [44]:
venues_details

,venue_id,venue_name,city,country,capacity
0,1438021,Namibia Cricket Ground,Windhoek,Namibia,None
1,363,Wanderers Cricket Ground,Windhoek,Namibia,None
2,1179,Udayana Cricket Ground,Bali,Indonesia,None
3,1528,Santarem Cricket Ground,Albergaria,Portugal,None
4,27,M.Chinnaswamy Stadium,Bengaluru,India,"40,000"
5,11,MA Chidambaram Stadium,Chennai,India,50000
6,80,Rajiv Gandhi International Stadium,Hyderabad,India,38000
7,81,Wankhede Stadium,Mumbai,India,"33,000"
8,485,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,India,50000
9,851,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,India,38000


## Players

In [50]:
players_list = []
match_ids = df["match_id"]

for mid in match_ids:
    score_url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{mid}/scard"
    res = requests.get(score_url, headers=headers)
    score_data = res.json()

    for inning in score_data.get('scorecard', []):
                    # -------------------------
            # 🏏 BATSMEN DATA
            # -------------------------
            for b in inning.get('batsman', []):
                players_list.append({
                    "player_id": b.get('id')})

            # -------------------------
            # 🏏 BOWLER DATA
            # -------------------------
            for bowler in inning.get('bowler', []):
                players_list.append({
                    "player_id": bowler.get('id')})

In [52]:
df1 = pd.DataFrame(players_list)

In [75]:
df1['player_id'].dropna().nunique()

686

In [76]:
players = []

player_ids = df1["player_id"].dropna().unique()

for pid in player_ids:
    score_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/{int(pid)}"
    res = requests.get(score_url, headers=headers)
    d = res.json()
    players.append({"id": d.get("id"),
            "name": d.get("name"),
            "role": d.get("role"),
            "batting": d.get("bat"),
            "bowling": d.get("bowl"),
            "country": d.get("birthPlace")})

KeyboardInterrupt: 

In [63]:
score_url = f"https://cricbuzz-cricket.p.rapidapi.com/stats/v1/player/6635"
res = requests.get(score_url, headers=headers)
d = res.json()

In [123]:
from datetime import datetime, timedelta

last_30_days = datetime.now() - timedelta(days=30)

result = df[df["start_date"] >= last_30_days]

# sort recent first
result = result.sort_values(by="start_date", ascending=False)

result

,match_id,series_id,series_name,match_desc,match_format,start_date,end_date,state,status,team1_id,...,team1_runs,team1_wkts,team1_overs,team2_runs,team2_wkts,team2_overs,venue_id,ground,city,result
50,153039,11766,"Women's Under-19 Tri-Series in Australia, 2026",3rd unofficial T20I,T20,2026-04-13 09:00:00,2026-04-13 12:30:00,Complete,Sri Lanka Women U19 won by 4 wkts,1328,...,91.0,9.0,19.6,92.0,6.0,19.3,1438034,Bill Pippen Oval,Gold Coast,Sri Lanka Women U19 won by 4 wkts
4,148492,11652,"Sweden tour of Indonesia, 2026",7th T20I,T20,2026-04-13 07:00:00,2026-04-13 10:30:00,Complete,Indonesia won by 4 wkts,534,...,145.0,9.0,19.6,148.0,6.0,17.4,1179,Udayana Cricket Ground,Bali,Indonesia won by 4 wkts
28,149160,11537,Pakistan Super League 2026,21st Match,T20,2026-04-12 19:30:00,2026-04-12 23:00:00,Complete,Hyderabad Kingsmen won by 6 wkts,330,...,153.0,9.0,19.6,157.0,4.0,18.1,24,National Stadium,Karachi,Hyderabad Kingsmen won by 6 wkts
19,149812,9241,Indian Premier League 2026,20th Match,T20,2026-04-12 19:30:00,2026-04-12 23:00:00,Complete,Royal Challengers Bengaluru won by 18 runs,59,...,240.0,4.0,19.6,222.0,5.0,19.6,81,Wankhede Stadium,Mumbai,Royal Challengers Bengaluru won by 18 runs
20,149801,9241,Indian Premier League 2026,19th Match,T20,2026-04-12 15:30:00,2026-04-12 19:00:00,Complete,Gujarat Titans won by 7 wkts,966,...,164.0,8.0,19.6,165.0,3.0,18.4,485,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,Gujarat Titans won by 7 wkts
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41,142043,11408,County Championship Division One 2026,3rd Match,TEST,2026-04-03 15:30:00,2026-04-06 22:30:00,Complete,Match drawn,149,...,302.0,10.0,120.3,226.0,10.0,60.2,62,Sophia Gardens,Cardiff,Match drawn
40,142065,11408,County Championship Division One 2026,5th Match,TEST,2026-04-03 15:30:00,2026-04-06 22:30:00,Complete,Match drawn,119,...,347.0,10.0,103.5,338.0,10.0,103.2,223,The Cooper Associates County Ground,Taunton,Match drawn
38,142054,11408,County Championship Division One 2026,4th Match,TEST,2026-04-03 15:30:00,2026-04-06 22:30:00,Complete,Sussex won by 222 runs,41,...,361.0,10.0,89.5,245.0,10.0,64.6,222,Grace Road,Leicester,Sussex won by 222 runs
37,142030,11408,County Championship Division One 2026,1st Match,TEST,2026-04-03 15:30:00,2026-04-06 22:30:00,Complete,Match drawn,148,...,328.0,10.0,95.1,544.0,10.0,128.3,20,Edgbaston,Birmingham,Match drawn


In [125]:
series_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/international"

res = requests.get(series_url, headers=headers)
series_data = res.json()

In [158]:
series = series_data.get("seriesMapProto", [])

row1 = []

for mt in series:
    series_type = mt.get("date")
    for j in mt.get("series", []):
        z = {"Date": series_type,
        "series_id": j.get("id"),
        "series_name": j.get("name"),
        "series_startdate": j.get("startDt"),
        "series_enddate": j.get("endDt")}
        
    row1.append(z)

In [159]:
series = series_data.get("seriesMapProto", [])

In [160]:
df2 = pd.DataFrame(row1)

In [68]:
df_series

,date,series_id,series_name,start_date,end_date
0,FEBRUARY 2024,7572,ICC Cricket World Cup League Two 2023-27,1707955200000,1830038400000
1,APRIL 2026,11652,"Sweden tour of Indonesia, 2026",1775520000000,1776038400000
2,APRIL 2026,11704,"Scotland tour of Namibia, 2026",1776211200000,1776470400000
3,APRIL 2026,11733,"New Zealand tour of Bangladesh, 2026",1776384000000,1777680000000
4,APRIL 2026,12012,"Luxembourg tour of France, 2026",1776988800000,1777161600000
5,MAY 2026,12034,"Gibraltar tour of Malta, 2026",1778112000000,1778284800000
6,MAY 2026,12004,ICC Men's T20 World Cup East Asia Pacific Qual...,1778198400000,1779062400000
7,JUNE 2026,11946,Sri Lanka tour of West Indies 2026,1780444800000,1783382400000
8,JUNE 2026,10559,"New Zealand tour of England, 2026",1780531200000,1782691200000
9,JUNE 2026,11641,Afghanistan tour of India 2026,1780704000000,1781913600000


In [161]:
df2

,Date,series_id,series_name,series_startdate,series_enddate
0,FEBRUARY 2024,7572,ICC Cricket World Cup League Two 2023-27,1707955200000,1806364800000
1,APRIL 2026,12012,"Luxembourg tour of France, 2026",1776988800000,1777161600000
2,MAY 2026,12004,ICC Men's T20 World Cup East Asia Pacific Qual...,1778198400000,1779062400000
3,JUNE 2026,10532,"India tour of England, 2026",1782864000000,1784419200000
4,JULY 2026,11991,India tour of Zimbabwe 2026,1784764800000,1785024000000
5,AUGUST 2026,10565,Pakistan tour of England 2026,1787097600000,1788912000000
6,SEPTEMBER 2026,11902,"West Indies tour of India, 2026",1790467200000,1792195200000
7,NOVEMBER 2026,11606,"Bangladesh tour of South Africa, 2026",1794700800000,1797120000000
8,DECEMBER 2026,11612,"England tour of South Africa, 2026-27",1797465600000,1799971200000
9,JANUARY 2027,11935,"Australia tour of India, 2027",1800489600000,1804032000000


In [5]:
import requests
import pandas as pd
from datetime import datetime


headers = {
    "X-RapidAPI-Key": "4005866697msh2b2c979bf694ec1p1cef3fjsne0259089faa3",
    "X-RapidAPI-Host": "cricbuzz-cricket.p.rapidapi.com"
}

# Step 1: Get all series
series_url = "https://cricbuzz-cricket.p.rapidapi.com/series/v1/international"
response = requests.get(series_url, headers=headers)
data = response.json()

series_map = data.get("seriesMapProto", {})

Total Series: 11
Empty DataFrame
Columns: []
Index: []
Total Matches: 0


In [70]:
import time

series_rows = []
for item in series_map:
    date = item.get("date")
    
    for series in item.get("series", []):
        series_rows.append({
            "date": date,
            "series_id": series.get("id"),
            "series_name": series.get("name"),
            "start_date": series.get("startDt"),
            "end_date": series.get("endDt")
        })

df_series = pd.DataFrame(series_rows)


Total Series: 30
Empty DataFrame
Columns: []
Index: []
Total Matches: 0


In [76]:
import requests
import pandas as pd

headers = {
    "X-RapidAPI-Key": "4005866697msh2b2c979bf694ec1p1cef3fjsne0259089faa3",
    "X-RapidAPI-Host": "cricbuzz-cricket.p.rapidapi.com"
}

url = "https://cricbuzz-cricket.p.rapidapi.com/matches/v1/recent"

response = requests.get(url, headers=headers)
data = response.json()

recent_matches = []

for match_type in data.get("typeMatches", []):
    for series in match_type.get("seriesMatches", []):
        
        series_info = series.get("seriesAdWrapper", {})
        series_id = series_info.get("seriesId")
        series_name = series_info.get("seriesName")
        
        for match in series_info.get("matches", []):
            info = match.get("matchInfo", {})
            
            recent_matches.append({
                "series_id": series_id,
                "series_name_recent": series_name,
                "match_id": info.get("matchId"),
                "team1": info.get("team1", {}).get("teamName"),
                "team2": info.get("team2", {}).get("teamName"),
                "match_format": info.get("matchFormat"),
                "status": info.get("status")
            })

df_recent = pd.DataFrame(recent_matches)

print(df_recent.head())


Empty DataFrame
Columns: []
Index: []


In [77]:
data

{'message': 'You have exceeded the MONTHLY quota for Requests on your current plan, BASIC. Upgrade your plan at https://rapidapi.com/cricketapilive/api/cricbuzz-cricket'}